# RNN Sentiment Analysis — IMDB

In [1]:
import re
from collections import Counter

from datasets import load_dataset

In [2]:
def tokenize(text):
    # lowercase, then pull out runs of word chars -> strips punctuation/HTML tags for free
    return re.findall(r"\b\w+\b", text.lower())

In [3]:
def build_vocab(texts, min_freq=2):
    '''
    This function takes whole text as input, tokenizes it and outputs a
    dict, that has the token as the key and the tokenid as the value.
    
    '''
    counts = Counter()
    for text in texts:
        counts.update(tokenize(text))

    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, freq in counts.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
    return vocab

In [4]:
def encode(text, vocab):
    return [vocab.get(token, vocab["<UNK>"]) for token in tokenize(text)]

In [5]:
imdb = load_dataset("stanfordnlp/imdb")
print(imdb)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})


In [6]:
sample = imdb["train"][0]
print("label:", sample["label"], "(0 = neg, 1 = pos)")
print("text:", sample["text"][:300], "...")

label: 0 (0 = neg, 1 = pos)
text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really h ...


In [7]:
vocab = build_vocab(imdb["train"]["text"])
print("vocab size:", len(vocab))

vocab size: 47010


In [8]:
encode(sample["text"], vocab)[:20]

[2, 3, 2, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 18]

In [9]:
import torch

In [10]:
def pad_batch(encoded_reviews, pad_value=0):
    '''
    Right-pads a batch of variable-length encoded reviews to the batch's own max length.

    Returns:
        padded: (batch_size, max_len) torch.long tensor of token ids
        lengths: (batch_size,) torch.long tensor of each review's real (unpadded) length
    '''
    lengths = torch.tensor([len(r) for r in encoded_reviews], dtype=torch.long)
    max_len = lengths.max().item()

    padded = torch.full((len(encoded_reviews), max_len), pad_value, dtype=torch.long)
    for i, review in enumerate(encoded_reviews):
        padded[i, :len(review)] = torch.tensor(review, dtype=torch.long)

    return padded, lengths

In [15]:
test_reviews = [imdb["train"][i]["text"] for i in range(3)]
test_encoded = [encode(text, vocab) for text in test_reviews]
print("lengths before padding:", [len(r) for r in test_encoded])

padded, lengths = pad_batch(test_encoded)
print("padded shape:", padded.shape)
print("lengths:", lengths)
print(padded[:, :15])  # first 15 columns of each row, to eyeball the padding

lengths before padding: [297, 228, 97]
padded shape: torch.Size([3, 297])
lengths: tensor([297, 228,  97])
tensor([[  2,   3,   2,   4,   5,   6,   7,   8,   9,  10,  11,  12,  13,  14,
          15],
        [  2,   4,   5,   6,  54,  42, 181,  91, 182, 183, 184,  18, 177, 178,
         185],
        [ 33, 207,  36, 291,  75,  38, 292,  12, 155,  23,  14, 293,  38, 155,
          54]])


In [16]:
embed_dim = 100
embedding = torch.nn.Embedding(num_embeddings=len(vocab), embedding_dim=embed_dim, padding_idx=vocab["<PAD>"])

In [17]:
embedded = embedding(padded)
print("padded shape:", padded.shape)
print("embedded shape:", embedded.shape)

padded shape: torch.Size([3, 297])
embedded shape: torch.Size([3, 297, 100])


In [18]:
hidden_dim = 128
batch_size = padded.shape[0]

h = torch.zeros(batch_size, hidden_dim)
print("h shape:", h.shape)

h shape: torch.Size([3, 128])


In [19]:
t = 0
x_t = embedded[:, t, :]
print("x_t shape:", x_t.shape)

x_t shape: torch.Size([3, 100])


In [20]:
W_xh = torch.randn(embed_dim, hidden_dim, requires_grad=True) * 0.01
W_hh = torch.randn(hidden_dim, hidden_dim, requires_grad=True) * 0.01
b_h = torch.zeros(hidden_dim, requires_grad=True)

print("W_xh:", W_xh.shape, "W_hh:", W_hh.shape, "b_h:", b_h.shape)

W_xh: torch.Size([100, 128]) W_hh: torch.Size([128, 128]) b_h: torch.Size([128])


In [21]:
h = torch.tanh(x_t @ W_xh + h @ W_hh + b_h)
print("h shape after one step:", h.shape)

h shape after one step: torch.Size([3, 128])
